In [1]:
import pandas as pd
import numpy as np
from collections import Counter
import ast
import os

#Load news.tsv
news_columns = [
    "news_id",
    "category",
    "subcategory",
    "title",
    "abstract",
    "url",
    "title_entities",
    "abstract_entities"
]

news_df = pd.read_csv("/kaggle/input/datasets/teshanlakruwan/mind-small-train/news.tsv",
    sep="\t",
    header=None,
    names=news_columns
)

print("News shape:", news_df.shape)
news_df.head()

#Load behaviors.tsv 

behaviors_columns = [
    "impression_id",
    "user_id",
    "time",
    "history",
    "impressions"
]

behaviors_df = pd.read_csv("/kaggle/input/datasets/teshanlakruwan/mind-small-train/behaviors.tsv",
    sep="\t",
    header=None,
    names=behaviors_columns
)

print("Behaviors shape:", behaviors_df.shape)
behaviors_df.head()

#Inspect caegories in the dataset
print("Unique categories:")
print(news_df["category"].unique())

print("\nCategory counts:")
print(news_df["category"].value_counts())

news_df["category_lower"] = news_df["category"].str.lower().str.strip()
news_df["subcategory_lower"] = news_df["subcategory"].str.lower().str.strip()

#Maping MIND categories to compatible with IAB content taxonomy 3.0 categories
mind_to_iab = {
    # Core categories
    "sports": "Sports",
    "finance": "Business_Finance",
    "autos": "Automotive",
    "travel": "Travel",
    "health": "Health",
    "lifestyle": "Lifestyle",
    "foodanddrink": "Food_Drink",

    # News-related
    "news": "General_News",
    "weather": "General_News",
    "middleeast": "General_News",
    "northamerica": "General_News",

    # Entertainment group
    "entertainment": "Entertainment",
    "tv": "Entertainment",
    "movies": "Entertainment",
    "music": "Entertainment",
    "video": "Entertainment",

    # Edge case
    "kids": "Lifestyle"   # or "Entertainment" (either is fine)
}

news_df["category_lower"] = news_df["category"].str.lower().str.strip()

def map_to_iab(category):
    return mind_to_iab.get(category, "Other")

news_df["iab_category"] = news_df["category_lower"].apply(map_to_iab)

print(news_df["iab_category"].value_counts())

#Remove Other type of categories if exists
news_df = news_df[news_df["iab_category"] != "Other"].copy()

#Validate If we got balanced datarecords for categories
print(news_df["iab_category"].value_counts())

#Get the news ids in behaviours to a list
def parse_history(history):
    if pd.isna(history) or history == "":
        return []
    return history.split()

behaviors_df["history_list"] = behaviors_df["history"].apply(parse_history)
behaviors_df["history_count"] = behaviors_df["history_list"].apply(len)

behaviors_df.head()


# Parse Impressions column data
def parse_impressions(impressions):
    result = []
    if pd.isna(impressions):
        return result
    
    for item in impressions.split():
        news_id, clicked = item.split("-")
        result.append((news_id, int(clicked)))
    return result

behaviors_df["impression_items"] = behaviors_df["impressions"].apply(parse_impressions)

behaviors_df.head()

#Expand Impressions
rows = []

for _, row in behaviors_df.iterrows():
    for news_id, clicked in row["impression_items"]:
        rows.append({
            "impression_id": row["impression_id"],
            "user_id": row["user_id"],
            "time": row["time"],
            "history_list": row["history_list"],
            "history_count": row["history_count"],
            "candidate_news_id": news_id,
            "clicked": clicked
        })

expanded_df = pd.DataFrame(rows)

print("Expanded shape:", expanded_df.shape)
expanded_df.head()


#Join refined two tables
master_df = expanded_df.merge(
    news_df,
    left_on="candidate_news_id",
    right_on="news_id",
    how="inner"
)

print("Master shape:", master_df.shape)
master_df.head()

#Add simple behavioural features
master_df["history_unique_count"] = master_df["history_list"].apply(lambda x: len(set(x)))

master_df["current_in_history"] = master_df.apply(
    lambda row: 1 if row["candidate_news_id"] in row["history_list"] else 0,
    axis=1
)

#Save Master dataset
master_df = master_df.sample(n=200000, random_state=42)
master_df.to_csv("/kaggle/working/master_train_dataset.csv", index=False)

News shape: (51282, 8)
Behaviors shape: (156965, 5)
Unique categories:
['lifestyle' 'health' 'news' 'sports' 'weather' 'entertainment' 'autos'
 'travel' 'foodanddrink' 'tv' 'finance' 'movies' 'video' 'music' 'kids'
 'middleeast' 'northamerica']

Category counts:
category
news             15774
sports           14510
finance           3107
foodanddrink      2551
lifestyle         2479
travel            2350
video             2068
weather           2048
health            1885
autos             1639
tv                 889
music              769
movies             606
entertainment      587
kids                17
middleeast           2
northamerica         1
Name: count, dtype: int64
iab_category
General_News        17825
Sports              14510
Entertainment        4919
Business_Finance     3107
Food_Drink           2551
Lifestyle            2496
Travel               2350
Health               1885
Automotive           1639
Name: count, dtype: int64
iab_category
General_News        17825

In [2]:
#Prepare Test dataset from Mind_small_dev dataset
#Load news.tsv
dev_news_df = pd.read_csv(
    "/kaggle/input/datasets/teshanlakruwan/mind-small-dev/news.tsv",
    sep="\t",
    header=None,
    names=news_columns
)

#Load behaviours.tsv
dev_behaviors_df = pd.read_csv(
    "/kaggle/input/datasets/teshanlakruwan/mind-small-dev/behaviors.tsv",
    sep="\t",
    header=None,
    names=behaviors_columns
)

dev_news_df = dev_news_df[[
    "news_id",
    "category",
    "subcategory",
    "title",
    "abstract"
]]

dev_news_df["category_lower"] = dev_news_df["category"].str.lower().str.strip()

dev_news_df["iab_category"] = dev_news_df["category_lower"].apply(map_to_iab)

dev_news_df = dev_news_df[dev_news_df["iab_category"] != "Other"].copy()

print(dev_news_df["iab_category"].value_counts())

#history
dev_behaviors_df["history_list"] = dev_behaviors_df["history"].apply(parse_history)
dev_behaviors_df["history_count"] = dev_behaviors_df["history_list"].apply(len)

#Impressions
dev_behaviors_df["impression_items"] = dev_behaviors_df["impressions"].apply(parse_impressions)

#expand Impressions
rows = []

for _, row in dev_behaviors_df.iterrows():
    for news_id, clicked in row["impression_items"]:
        rows.append({
            "impression_id": row["impression_id"],
            "user_id": row["user_id"],
            "time": row["time"],
            "history_list": row["history_list"],
            "history_count": row["history_count"],
            "candidate_news_id": news_id,
            "clicked": clicked
        })

dev_expanded_df = pd.DataFrame(rows)

print(dev_expanded_df.shape)
dev_expanded_df.head()

#join with news test data
master_dev_df = dev_expanded_df.merge(
    dev_news_df,
    left_on="candidate_news_id",
    right_on="news_id",
    how="inner"
)

#Combine behavioural features

print(master_dev_df.shape)
master_dev_df.head()

master_dev_df["history_unique_count"] = master_dev_df["history_list"].apply(lambda x: len(set(x)))

master_dev_df["current_in_history"] = master_dev_df.apply(
    lambda row: 1 if row["candidate_news_id"] in row["history_list"] else 0,
    axis=1
)

#save data
master_dev_df = master_dev_df.sample(n=50000, random_state=42)
master_dev_df.to_csv("/kaggle/working/master_dev_dataset.csv", index=False)

iab_category
General_News        14507
Sports              11760
Entertainment        4144
Business_Finance     2563
Food_Drink           2248
Lifestyle            2143
Travel               1845
Health               1715
Automotive           1490
Name: count, dtype: int64
(2740998, 7)
(2740998, 14)


In [3]:
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer


#Fill missing textfields
for df in [master_df, master_dev_df]:
    df["title"] = df["title"].fillna("")
    df["abstract"] = df["abstract"].fillna("")
    df["subcategory"] = df["subcategory"].fillna("")

    # Combine text fields into one column
    df["text"] = (
        df["title"].astype(str) + " " +
        df["abstract"].astype(str) + " " +
        df["subcategory"].astype(str)
    ).str.strip()

print(master_df[["text", "iab_category"]].head())
print(master_dev_df[["text", "iab_category"]].head())

# Labels
y_train = master_df["iab_category"]
y_dev = master_dev_df["iab_category"]

# Text input
X_train_text_raw = master_df["text"]
X_dev_text_raw = master_dev_df["text"]

# Behaviour features
behaviour_cols = [
    "history_count",
    "history_unique_count",
    "current_in_history"
]

X_train_behaviour = master_df[behaviour_cols].copy()
X_dev_behaviour = master_dev_df[behaviour_cols].copy()

print("Train labels shape:", y_train.shape)
print("Dev labels shape:", y_dev.shape)
print("Train behaviour shape:", X_train_behaviour.shape)
print("Dev behaviour shape:", X_dev_behaviour.shape)


#TF-IDF Vectorization

tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=5
)

X_train_text = tfidf.fit_transform(X_train_text_raw)
X_dev_text = tfidf.transform(X_dev_text_raw)

print("TF-IDF train shape:", X_train_text.shape)
print("TF-IDF dev shape:", X_dev_text.shape)

#Encode labels


label_encoder = LabelEncoder()

y_train_enc = label_encoder.fit_transform(y_train)
y_dev_enc = label_encoder.transform(y_dev)

print("Classes:", label_encoder.classes_)


                                                      text      iab_category
3874367  Carrie Underwood rocks CMAs and several gorgeo...         Lifestyle
4422490  Kodak Black Sentenced to Over 3 Years in Priso...     Entertainment
147236   'Priceless' finds that turned out to be worthl...  Business_Finance
5023576  The highlights and lowlights from the world of...     Entertainment
5455306  Groom Makes The Most Heartfelt Vows To His 9-Y...         Lifestyle
                                                      text   iab_category
1371415  How much turkey do you need to buy per person?...     Food_Drink
559975   At NATO summit, Trump to stress US allies' def...   General_News
2488017  Wrecked Toyota Supra Launch Edition Didn't Eve...     Automotive
1258873  New Ant Species Discovered in Ant Expert's Bac...  Entertainment
247757   Months-Long 60 Swarm Construction To Conclude ...     Automotive
Train labels shape: (200000,)
Dev labels shape: (50000,)
Train behaviour shape: (200000, 3)
De

In [4]:
# Test text only prediction test
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

text_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

text_model.fit(X_train_text, y_train_enc)

y_pred_text = text_model.predict(X_dev_text)

acc = accuracy_score(y_dev_enc, y_pred_text)
prec, rec, f1, _ = precision_recall_fscore_support(
    y_dev_enc, y_pred_text, average="weighted"
)

print("=== Text Only Results ===")
print("Accuracy :", acc)
print("Precision:", prec)
print("Recall   :", rec)
print("F1-score :", f1)

print("\nDetailed Report:")
print(classification_report(
    y_dev_enc,
    y_pred_text,
    target_names=label_encoder.classes_
))

=== Text Only Results ===
Accuracy : 0.9596
Precision: 0.9612984747585197
Recall   : 0.9596
F1-score : 0.9590425023890318

Detailed Report:
                  precision    recall  f1-score   support

      Automotive       0.91      0.96      0.94      2024
Business_Finance       1.00      1.00      1.00      4096
   Entertainment       0.98      0.99      0.98      9779
      Food_Drink       0.95      1.00      0.97      3863
    General_News       0.92      1.00      0.96     12533
          Health       1.00      0.94      0.97      2478
       Lifestyle       0.98      0.92      0.95      6505
          Sports       0.97      0.91      0.94      6410
          Travel       1.00      0.77      0.87      2312

        accuracy                           0.96     50000
       macro avg       0.97      0.94      0.95     50000
    weighted avg       0.96      0.96      0.96     50000



In [5]:
#behaviours only prdiction test
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_behaviour_scaled = scaler.fit_transform(X_train_behaviour)
X_dev_behaviour_scaled = scaler.transform(X_dev_behaviour)

behaviour_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

behaviour_model.fit(X_train_behaviour_scaled, y_train_enc)

y_pred_behaviour = behaviour_model.predict(X_dev_behaviour_scaled)

acc = accuracy_score(y_dev_enc, y_pred_behaviour)
prec, rec, f1, _ = precision_recall_fscore_support(
    y_dev_enc, y_pred_behaviour, average="weighted"
)

print("=== Behaviour Only Results ===")
print("Accuracy :", acc)
print("Precision:", prec)
print("Recall   :", rec)
print("F1-score :", f1)

print("\nDetailed Report:")
print(classification_report(
    y_dev_enc,
    y_pred_behaviour,
    target_names=label_encoder.classes_
))

=== Behaviour Only Results ===
Accuracy : 0.15558
Precision: 0.12790090020422168
Recall   : 0.15558
F1-score : 0.08048990427955524

Detailed Report:
                  precision    recall  f1-score   support

      Automotive       0.00      0.00      0.00      2024
Business_Finance       0.00      0.00      0.00      4096
   Entertainment       0.19      0.63      0.30      9779
      Food_Drink       0.08      0.29      0.12      3863
    General_News       0.27      0.00      0.00     12533
          Health       0.00      0.00      0.00      2478
       Lifestyle       0.00      0.00      0.00      6505
          Sports       0.13      0.07      0.09      6410
          Travel       0.00      0.00      0.00      2312

        accuracy                           0.16     50000
       macro avg       0.07      0.11      0.06     50000
    weighted avg       0.13      0.16      0.08     50000



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/m

In [6]:
#Textual + Behavioural Prdiction test
from scipy.sparse import hstack, csr_matrix

X_train_combined = hstack([
    X_train_text,
    csr_matrix(X_train_behaviour_scaled)
])

X_dev_combined = hstack([
    X_dev_text,
    csr_matrix(X_dev_behaviour_scaled)
])

combined_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

combined_model.fit(X_train_combined, y_train_enc)

y_pred_combined = combined_model.predict(X_dev_combined)

acc = accuracy_score(y_dev_enc, y_pred_combined)
prec, rec, f1, _ = precision_recall_fscore_support(
    y_dev_enc, y_pred_combined, average="weighted"
)

print("=== Text + Behaviour Results ===")
print("Accuracy :", acc)
print("Precision:", prec)
print("Recall   :", rec)
print("F1-score :", f1)

print("\nDetailed Report:")
print(classification_report(
    y_dev_enc,
    y_pred_combined,
    target_names=label_encoder.classes_
))

=== Text + Behaviour Results ===
Accuracy : 0.9574
Precision: 0.9589181240691995
Recall   : 0.9574
F1-score : 0.9568494662080478

Detailed Report:
                  precision    recall  f1-score   support

      Automotive       0.98      0.95      0.96      2024
Business_Finance       1.00      1.00      1.00      4096
   Entertainment       0.97      0.97      0.97      9779
      Food_Drink       0.95      1.00      0.97      3863
    General_News       0.92      1.00      0.96     12533
          Health       1.00      0.96      0.98      2478
       Lifestyle       0.98      0.92      0.95      6505
          Sports       0.95      0.91      0.93      6410
          Travel       1.00      0.77      0.87      2312

        accuracy                           0.96     50000
       macro avg       0.97      0.94      0.95     50000
    weighted avg       0.96      0.96      0.96     50000



In [7]:
# Compare 3 approches and evaluate it
results = []

def evaluate_model(name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted"
    )
    results.append({
        "Model": name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1_score": f1
    })

evaluate_model("Text Only", y_dev_enc, y_pred_text)
evaluate_model("Behaviour Only", y_dev_enc, y_pred_behaviour)
evaluate_model("Text + Behaviour", y_dev_enc, y_pred_combined)

results_df = pd.DataFrame(results)
print(results_df)

              Model  Accuracy  Precision   Recall  F1_score
0         Text Only   0.95960   0.961298  0.95960  0.959043
1    Behaviour Only   0.15558   0.127901  0.15558  0.080490
2  Text + Behaviour   0.95740   0.958918  0.95740  0.956849


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [8]:
#Create ad pool using Ads_Creative_Text_Programmatic dataset

import pandas as pd

# =========================================
# 1. Load ad dataset from Hugging Face
# =========================================
ads_df = pd.read_csv(
    "hf://datasets/PeterBrendan/Ads_Creative_Text_Programmatic/ads_creative_text_sample.csv"
)

# Rename the text column
ads_df = ads_df.rename(columns={"text": "ad_text"})

# Clean ad text
ads_df["ad_text"] = ads_df["ad_text"].fillna("").astype(str).str.lower().str.strip()

print("Raw ads shape:", ads_df.shape)
print(ads_df.head())


# =========================================
# 2. Remove empty ads
# =========================================
ads_df = ads_df[ads_df["ad_text"] != ""].copy()

# Optional: remove very short ad texts
ads_df = ads_df[ads_df["ad_text"].str.len() > 10].copy()

print("After cleaning shape:", ads_df.shape)


# =========================================
# 3. Categorize ads using Stage 1 TF-IDF + model
#    IMPORTANT:
#    - tfidf = your trained Stage 1 vectorizer
#    - text_model = your trained Stage 1 classifier
#    - label_encoder = your fitted label encoder
# =========================================
ads_tfidf = tfidf.transform(ads_df["ad_text"])

ads_pred_idx = text_model.predict(ads_tfidf)
ads_pred_prob = text_model.predict_proba(ads_tfidf)

ads_df["category"] = label_encoder.inverse_transform(ads_pred_idx)
ads_df["category_confidence"] = ads_pred_prob.max(axis=1)

print("\nPredicted category distribution:")
print(ads_df["category"].value_counts())


# =========================================
# 4. Add ad type
#    All classified ads are targeted for now
# =========================================
ads_df["type"] = "targeted"


# =========================================
# 5. Add ad_id
# =========================================
ads_df = ads_df.reset_index(drop=True)
ads_df["ad_id"] = ["AD_" + str(i).zfill(5) for i in range(len(ads_df))]


# =========================================
# 6. Keep only needed columns
# =========================================
ads_pool_df = ads_df[[
    "ad_id",
    "ad_text",
    "category",
    "category_confidence",
    "type"
]].copy()


# =========================================
# 7. Add some generic fallback ads manually
# =========================================
generic_ads = pd.DataFrame({
    "ad_id": ["GEN_001", "GEN_002", "GEN_003", "GEN_004", "GEN_005"],
    "ad_text": [
        "discover great offers today",
        "find products and services that match your needs",
        "explore trusted brands and new deals",
        "see useful recommendations for everyday life",
        "check out featured offers available now"
    ],
    "category": ["General", "General", "General", "General", "General"],
    "category_confidence": [1.0, 1.0, 1.0, 1.0, 1.0],
    "type": ["generic", "generic", "generic", "generic", "generic"]
})


# =========================================
# 8. Combine targeted + generic ads
# =========================================
ads_pool_df = pd.concat([ads_pool_df, generic_ads], ignore_index=True)

print("\nFinal ad pool shape:", ads_pool_df.shape)
print(ads_pool_df.head())


# =========================================
# 9. Optional: inspect category counts
# =========================================
print("\nFinal ad pool category distribution:")
print(ads_pool_df["category"].value_counts())

print("\nFinal ad pool type distribution:")
print(ads_pool_df["type"].value_counts())


# =========================================
# 10. Save to CSV
# =========================================
ads_pool_df.to_csv("/kaggle/working/ads_pool.csv", index=False)

print("\nSaved file: /kaggle/working/ads_pool.csv")



Raw ads shape: (1000, 2)
                                             ad_text  dimensions
0  up to\n$100 off\nroundtrip\nflights to\nirelan...  (160, 600)
1  yp the real\nур\nyellow pages\nfind cheap\ngas...  (300, 250)
2  food navigator\nusa\nplant-based meat:\nbeyond...  (300, 600)
3  monstrous\nairflow\n$20-$23\nduramax\nbanks\n+...  (300, 250)
4  yummy\ncombs\n*\na\nnutriti\nwellne\nnow\nsafe...   (728, 90)
After cleaning shape: (999, 2)

Predicted category distribution:
category
Lifestyle           252
General_News        234
Entertainment       183
Sports              103
Business_Finance     56
Health               54
Automotive           42
Travel               40
Food_Drink           35
Name: count, dtype: int64

Final ad pool shape: (1004, 5)
      ad_id                                            ad_text       category  \
0  AD_00000  up to\n$100 off\nroundtrip\nflights to\nirelan...         Travel   
1  AD_00001  yp the real\nур\nyellow pages\nfind cheap\ngas...   General_Ne